In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# CELL A01
# Structural Grid Features — Analysis Copy (Read-Only)

from collections import deque, Counter

def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else (0, 0)

def infer_background_color(grid):
    flat = [c for row in grid for c in row]
    return Counter(flat).most_common(1)[0][0] if flat else 0

def grid_colors(grid):
    return sorted(set(c for row in grid for c in row))

def connected_components(grid, background):
    h, w = grid_shape(grid)
    visited = [[False]*w for _ in range(h)]
    components = []

    for r in range(h):
        for c in range(w):
            if visited[r][c] or grid[r][c] == background:
                continue
            color = grid[r][c]
            q = deque([(r,c)])
            visited[r][c] = True
            cells = []

            while q:
                cr, cc = q.popleft()
                cells.append((cr,cc))
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc = cr+dr, cc+dc
                    if 0<=nr<h and 0<=nc<w and not visited[nr][nc] and grid[nr][nc]==color:
                        visited[nr][nc] = True
                        q.append((nr,nc))

            rows = [p[0] for p in cells]
            cols = [p[1] for p in cells]
            components.append({
                "color": color,
                "cells": tuple(cells),
                "area": len(cells),
                "bbox": (min(rows), max(rows), min(cols), max(cols)),
            })

    components.sort(key=lambda c: (c["bbox"][0], c["bbox"][2], c["color"]))
    return components

def is_vertical_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[r][w-1-c] for r in range(h) for c in range(w))

def is_horizontal_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[h-1-r][c] for r in range(h) for c in range(w))

def is_rot180_symmetric(grid):
    h,w = grid_shape(grid)
    return all(grid[r][c]==grid[h-1-r][w-1-c] for r in range(h) for c in range(w))

def grid_features(grid):
    bg = infer_background_color(grid)
    comps = connected_components(grid, bg)
    return {
        "shape": grid_shape(grid),
        "background": bg,
        "colors": grid_colors(grid),
        "num_colors": len(grid_colors(grid)),
        "num_components": len(comps),
        "components": comps,
        "symmetry": {
            "vertical": is_vertical_symmetric(grid),
            "horizontal": is_horizontal_symmetric(grid),
            "rot180": is_rot180_symmetric(grid),
        },
    }

In [ ]:
# CELL A02
# Hypothesis Scoring — Analysis Copy (Read-Only, v1.1)

"""
Structural scoring for solver hypotheses.

INVARIANTS:
- Deterministic
- Purely symbolic
- No learning
- Analysis-only
"""

def score_hypothesis(hypothesis):
    feats = grid_features(hypothesis.grid)
    score = 0.0

    # ------------------------------------------------------------------
    # 1. Component complexity (primary signal)
    # ------------------------------------------------------------------
    score += feats["num_components"] * 5.0

    # ------------------------------------------------------------------
    # 2. Spatial footprint (secondary signal)
    # ------------------------------------------------------------------
    for comp in feats["components"]:
        r0, r1, c0, c1 = comp["bbox"]
        score += (r1 - r0 + 1) * (c1 - c0 + 1) * 0.1

    # ------------------------------------------------------------------
    # 3. Color complexity
    # ------------------------------------------------------------------
    score += feats["num_colors"] * 2.0

    # ------------------------------------------------------------------
    # 4. Symmetry bonuses (weak, but stabilizing)
    # ------------------------------------------------------------------
    if feats["symmetry"]["vertical"]:
        score -= 1.0
    if feats["symmetry"]["horizontal"]:
        score -= 1.0
    if feats["symmetry"]["rot180"]:
        score -= 0.5

    # ------------------------------------------------------------------
    # 5. NEW: Repeated-operator penalty (breaks symmetry)
    # ------------------------------------------------------------------
    history = hypothesis.history
    for i in range(1, len(history)):
        if history[i] == history[i - 1]:
            score += 3.0   # discourage rotate→rotate, etc.

    # ------------------------------------------------------------------
    # 6. NEW: No-op structural penalty
    # ------------------------------------------------------------------
    # If nothing structural changed, penalize lightly
    if feats["num_components"] == grid_features(hypothesis.grid)["num_components"]:
        score += 1.0

    return round(score, 6)

In [ ]:
# CELL A02a
# Symbolic Solver Search Core (Analysis-Only)

"""
Defines the symbolic solver search core for the analysis notebook.

INVARIANTS:
- Analysis-only
- Deterministic
- No executors
- No governance
- No submission
"""

from dataclasses import dataclass
from typing import Tuple, List

# -----------------------------------------------------------------------------
# Solver hypothesis container
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class SolverHypothesis:
    grid: List[List[int]]
    history: Tuple[str, ...]


# -----------------------------------------------------------------------------
# Minimal operator set (analysis-safe)
# -----------------------------------------------------------------------------

def rotate90(grid):
    return [list(row) for row in zip(*grid[::-1])]

def flip_h(grid):
    return [row[::-1] for row in grid]

def normalize_grid(grid):
    return grid


def _solver_operators():
    return [rotate90, flip_h]


# -----------------------------------------------------------------------------
# Expansion + pruning (bounded)
# -----------------------------------------------------------------------------

def expand_and_prune(
    hypotheses: List[SolverHypothesis],
    operators,
    max_hypotheses: int,
):
    expanded = []

    for h in hypotheses:
        for op in operators:
            try:
                new_grid = op(h.grid)
            except Exception:
                continue

            expanded.append(
                SolverHypothesis(
                    grid=new_grid,
                    history=h.history + (op.__name__,),
                )
            )

    if not expanded:
        return hypotheses

    expanded.sort(key=score_hypothesis)
    return expanded[:max_hypotheses]


# -----------------------------------------------------------------------------
# Solver search entry point (analysis-only)
# -----------------------------------------------------------------------------

def run_search(
    input_grid,
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Runs a bounded symbolic search.

    Returns:
    {
        "hypotheses": [SolverHypothesis, ...]
    }
    """

    hypotheses = [
        SolverHypothesis(
            grid=normalize_grid(input_grid),
            history=(),
        )
    ]

    operators = _solver_operators()

    for _ in range(max_steps):
        hypotheses = expand_and_prune(
            hypotheses,
            operators,
            max_hypotheses,
        )

    hypotheses.sort(key=score_hypothesis)

    return {
        "hypotheses": hypotheses,
    }


print("[ANALYSIS] ✅ Symbolic solver search core defined")


In [ ]:
# CELL A03
# Solver Trace Store (Analysis-Only)

"""
Defines the global solver trace store for the analysis notebook.

INVARIANTS:
- Analysis-only
- Append-only
- Deterministic
- No execution, no governance, no submission
"""

# -----------------------------------------------------------------------------
# Trace store initialization
# -----------------------------------------------------------------------------

if "traces" not in globals():
    traces = []

print(f"[ANALYSIS] ✅ Solver trace store ready ({len(traces)} traces)")

In [ ]:
# CELL A04
# ARC Dataset Loader (Analysis-Only, ARC-AGI-2 2026)

"""
Loads ARC-AGI-2 datasets for solver analysis.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- NO execution
- NO governance
- NO submission logic
"""

import json
import os

# -----------------------------------------------------------------------------
# ARC-AGI-2 Kaggle competition paths (2026)
# -----------------------------------------------------------------------------

ARC_DATA_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

TRAIN_PATH = os.path.join(ARC_DATA_DIR, "arc-agi_training_challenges.json")
EVAL_PATH  = os.path.join(ARC_DATA_DIR, "arc-agi_evaluation_challenges.json")

# -----------------------------------------------------------------------------
# Defensive existence checks
# -----------------------------------------------------------------------------

for path in [TRAIN_PATH, EVAL_PATH]:
    if not os.path.exists(path):
        raise RuntimeError(
            f"ARC dataset file not found: {path}\n"
            "Kaggle dataset layout may have changed."
        )

# -----------------------------------------------------------------------------
# Load datasets
# -----------------------------------------------------------------------------

with open(TRAIN_PATH, "r") as f:
    train_challenges = json.load(f)

with open(EVAL_PATH, "r") as f:
    eval_challenges = json.load(f)

# -----------------------------------------------------------------------------
# ARC namespace (analysis-only container)
# -----------------------------------------------------------------------------

class ARC:
    train_challenges = train_challenges
    eval_challenges = eval_challenges

print(
    "[ARC] ✅ ARC-AGI-2 datasets loaded (analysis-only)\n"
    f" - Train tasks: {len(ARC.train_challenges)}\n"
    f" - Eval tasks: {len(ARC.eval_challenges)}"
)

In [ ]:
# CELL A05
# Solver Traced Entry Point (Analysis-Only)

"""
Defines run_search_traced for the analysis notebook.

INVARIANTS:
- Analysis-only
- No executors
- No governance
- No submission
- Observational tracing only
"""

# -----------------------------------------------------------------------------
# Dependency check
# -----------------------------------------------------------------------------

required = [
    "run_search",
    "traces",
    "grid_features",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "run_search_traced cannot be defined.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Traced wrapper
# -----------------------------------------------------------------------------

def run_search_traced(
    input_grid,
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Runs the symbolic solver and records a trace.

    This function is observational ONLY.
    """

    result = run_search(
        input_grid=input_grid,
        max_steps=max_steps,
        max_hypotheses=max_hypotheses,
    )

    hypotheses = result.get("hypotheses", [])

    trace = {
        "input_grid": input_grid,
        "input_features": grid_features(input_grid),
        "search_config": {
            "max_steps": max_steps,
            "max_hypotheses": max_hypotheses,
        },
        "num_final_hypotheses": len(hypotheses),
        "raw_hypotheses": hypotheses,
    }

    traces.append(trace)
    return result


print("[ANALYSIS] ✅ run_search_traced defined (analysis-only)")

In [ ]:
# CELL D00
# Solver Core Bootstrap (Analysis / Dataset Notebook)

"""
This cell defines the REQUIRED solver-core symbols for the
solver-derivation notebook.

This notebook is ANALYSIS-ONLY.
It must never define or modify solver behavior.

Expected source:
- Solver core cells copied from submission notebook
  (grid_features, score_hypothesis, run_search_traced, etc.)

This cell performs:
- Explicit dependency validation
- Trace store initialization (if absent)
"""

# -----------------------------------------------------------------------------
# Required solver-core symbols
# -----------------------------------------------------------------------------

REQUIRED_SOLVER_SYMBOLS = [
    "grid_features",
    "score_hypothesis",
]

missing = [s for s in REQUIRED_SOLVER_SYMBOLS if s not in globals()]

if missing:
    raise RuntimeError(
        "Solver core not loaded in solver-derivation notebook.\n\n"
        "You must COPY the solver core cells (structural features, "
        "scoring, search, tracing) into this notebook BEFORE running "
        "dataset generation.\n\n"
        "Missing symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Trace store initialization (analysis-only)
# -----------------------------------------------------------------------------

if "traces" not in globals():
    traces = []
    print("[BOOTSTRAP] Initialized empty solver trace store.")

print(
    "[BOOTSTRAP] ✅ Solver core detected for dataset derivation.\n"
    f" - grid_features: {'grid_features' in globals()}\n"
    f" - score_hypothesis: {'score_hypothesis' in globals()}\n"
    f" - traces available: {len(traces)}"
)

In [ ]:
# CELL D00.1
# Bulk Solver Trace Generation (Analysis-Only, Deterministic)

"""
This cell populates the solver trace store by running the
symbolic solver in advisory / traced mode over ARC inputs.

INVARIANTS:
- Analysis-only
- Offline
- Deterministic
- No executor, no governance, no submission
"""

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "ARC",
    "run_search_traced",
    "traces",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Bulk trace generation failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Configuration (SAFE DEFAULTS)
# -----------------------------------------------------------------------------

MAX_TASKS = 50
MAX_STEPS = 2
MAX_HYPOTHESES = 16

SOURCE_SPLIT = "train"

# -----------------------------------------------------------------------------
# Select ARC source
# -----------------------------------------------------------------------------

if SOURCE_SPLIT == "train":
    source = ARC.train_challenges
elif SOURCE_SPLIT == "eval":
    source = ARC.eval_challenges
else:
    raise RuntimeError(f"Unknown SOURCE_SPLIT: {SOURCE_SPLIT}")

# -----------------------------------------------------------------------------
# Trace generation
# -----------------------------------------------------------------------------

initial_trace_count = len(traces)
processed = 0

for task_id, task in source.items():
    if processed >= MAX_TASKS:
        break

    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0]["input"]

    run_search_traced(
        input_grid=input_grid,
        max_steps=MAX_STEPS,
        max_hypotheses=MAX_HYPOTHESES,
    )

    traces[-1]["task_id"] = task_id
    traces[-1]["split"] = SOURCE_SPLIT

    processed += 1

print(
    "[TRACE] ✅ Bulk solver trace generation complete\n"
    f" - Source split: {SOURCE_SPLIT}\n"
    f" - Tasks processed: {processed}\n"
    f" - Traces before: {initial_trace_count}\n"
    f" - Traces after: {len(traces)}"
)

In [ ]:
# CELL D01
# Solver-Derivation Dataset Generation (v1.0, Observational Only)

"""
Materializes a solver-derivation dataset from recorded solver traces.

INVARIANTS:
- Offline only
- Deterministic
- Observational (non-causal)
- Analysis notebook ONLY
"""

import json
import hashlib
from datetime import datetime, timezone

# -----------------------------------------------------------------------------
# Helper utilities
# -----------------------------------------------------------------------------

def _stable_hash(obj) -> str:
    blob = json.dumps(obj, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def _extract_hypothesis_record(h, rank: int):
    feats = grid_features(h.grid)
    return {
        "rank": rank,
        "score": score_hypothesis(h),
        "history": list(h.history),
        "grid": h.grid,
        "features": {
            "shape": feats["shape"],
            "background": feats["background"],
            "num_colors": feats["num_colors"],
            "num_components": feats["num_components"],
            "symmetry": feats["symmetry"],
        },
    }


# -----------------------------------------------------------------------------
# Dataset construction
# -----------------------------------------------------------------------------

records = []

for idx, trace in enumerate(traces):
    record = {
        "record_id": None,
        "task_id": trace.get("task_id", f"UNKNOWN_{idx}"),
        "split": trace.get("split", "unknown"),
        "input": {
            "grid": trace.get("input_grid"),
            "features": trace.get("input_features"),
        },
        "solver_run": trace.get("search_config"),
        "hypotheses": [
            _extract_hypothesis_record(h, rank=i + 1)
            for i, h in enumerate(trace.get("raw_hypotheses", []))
        ],
        "summary": {
            "num_hypotheses_retained": trace.get("num_final_hypotheses"),
        },
    }

    record["record_id"] = _stable_hash(record)
    records.append(record)

# -----------------------------------------------------------------------------
# Final dataset wrapper
# -----------------------------------------------------------------------------

dataset = {
    "dataset_version": "v1.0",
    "schema_version": "solver-derivation-v1",
    "generated_by": "arc-agi-v17-solver-derivation.ipynb",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "generation_context": {
        "deterministic": True,
        "offline_only": True,
        "solver_version": "v17-symbolic-core",
        "executor_used": False,
    },
    "records": records,
}

# -----------------------------------------------------------------------------
# Write dataset to disk
# -----------------------------------------------------------------------------

OUTPUT_PATH = "solver_derivation_dataset_v1.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

print(
    "[DATASET] ✅ Solver-derivation dataset written\n"
    f" - Path: {OUTPUT_PATH}\n"
    f" - Records: {len(records)}\n"
    f" - Schema: solver-derivation-v1"
)

In [ ]:
# CELL D01a
# Executor Output Loader (Analysis-Only, Dataset 2 Infrastructure)

"""
Loads executor outputs for solver–executor disagreement analysis.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution or submission

This cell is robust to Kaggle's dataset mounting layout:
- /kaggle/input/datasets/<dataset>/...
- /kaggle/input/competitions/<competition>/...
"""

import json
import os

KAGGLE_INPUT_ROOT = "/kaggle/input"
TARGET_FILENAME = "executor_outputs.json"

executor_output_path = None

# -----------------------------------------------------------------------------
# RECURSIVE SEARCH UNDER /kaggle/input
# -----------------------------------------------------------------------------
for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
    if TARGET_FILENAME in files:
        executor_output_path = os.path.join(root, TARGET_FILENAME)
        break

if executor_output_path is None:
    raise RuntimeError(
        "[D01a] executor_outputs.json not found under /kaggle/input.\n\n"
        "Searched recursively under:\n"
        "  /kaggle/input\n\n"
        "Fix:\n"
        "  Ensure the dataset containing executor_outputs.json is attached\n"
        "  and visible in the notebook side panel."
    )

# -----------------------------------------------------------------------------
# LOAD EXECUTOR OUTPUTS
# -----------------------------------------------------------------------------
with open(executor_output_path, "r") as f:
    executor_outputs = json.load(f)

print(
    "[D01a] ✅ Executor outputs loaded (analysis-only)\n"
    f" - Path  : {executor_output_path}\n"
    f" - Tasks : {len(executor_outputs)}"
)

In [ ]:
# CELL D01a.1
# Solver Trace Normalizer (Analysis-Only Infrastructure)

"""
Normalizes solver behavior into `solver_traces`.

Priority:
1) Use Dataset‑1 `records` (solver-derivation dataset) if present:
   - final_output := top-ranked hypothesis grid (rank 1)
   - top_k_grids  := all retained hypothesis grids
   - scores       := hypothesis scores

2) Fallback to in-notebook `traces`:
   - expects list of trace dicts with task_id and optional output/prediction

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution
"""

from datetime import datetime, timezone

_CELL_D01A_1_SOURCE = r'''
# CELL D01a.1
# Solver Trace Normalizer

from datetime import datetime, timezone

solver_traces = {}

if "records" in globals() and isinstance(records, list) and records:
    for rec in records:
        task_id = rec.get("task_id")
        hyps = rec.get("hypotheses", [])
        top_grid = hyps[0]["grid"] if hyps else None
        solver_traces[task_id] = {
            "final_output": top_grid,
            "top_k_grids": [h["grid"] for h in hyps],
            "scores": [h.get("score") for h in hyps],
            "raw": rec,
        }
else:
    # fallback on traces list
    ...
'''

solver_traces = {}

source_used = None
normalized = 0
skipped = 0

# ---------------------------------------------------------------------
# Preferred source: Dataset‑1 `records` (produced by CELL D01)
# ---------------------------------------------------------------------
if "records" in globals() and isinstance(records, list) and len(records) > 0:
    source_used = "records(list)"

    for rec in records:
        if not isinstance(rec, dict):
            skipped += 1
            continue

        task_id = rec.get("task_id")
        if not task_id:
            skipped += 1
            continue

        hyps = rec.get("hypotheses", [])
        if not isinstance(hyps, list):
            hyps = []

        top_k_grids = []
        top_k_scores = []

        for h in hyps:
            if isinstance(h, dict) and "grid" in h:
                top_k_grids.append(h["grid"])
                top_k_scores.append(h.get("score"))

        final_output = top_k_grids[0] if top_k_grids else None

        solver_traces[task_id] = {
            "final_output": final_output,      # canonical "solver choice" for Dataset-2
            "top_k_grids": top_k_grids,        # all retained hypotheses for match-rank
            "top_k_scores": top_k_scores,      # aligned with grids
            "raw": rec,                        # full record for auditability
        }
        normalized += 1

# ---------------------------------------------------------------------
# Fallback source: `traces` (often list-of-trace dicts)
# ---------------------------------------------------------------------
elif "traces" in globals():
    source_used = f"traces({type(traces).__name__})"

    if isinstance(traces, list):
        for idx, entry in enumerate(traces):
            if not isinstance(entry, dict):
                skipped += 1
                continue

            task_id = entry.get("task_id") or entry.get("task") or entry.get("id")
            if not task_id:
                skipped += 1
                continue

            final_output = (
                entry.get("final_output")
                or entry.get("output")
                or entry.get("prediction")
            )

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],         # unknown in this format
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1

    elif isinstance(traces, dict):
        for task_id, entry in traces.items():
            if isinstance(entry, dict):
                final_output = (
                    entry.get("final_output")
                    or entry.get("output")
                    or entry.get("prediction")
                )
            else:
                final_output = entry

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1
    else:
        raise RuntimeError(
            f"[D01a.1] Unsupported `traces` type: {type(traces).__name__}"
        )

else:
    raise RuntimeError(
        "[D01a.1] No solver trace source found.\n"
        "Expected `records` (preferred) or `traces` (fallback) to exist."
    )

D01A_1_AUDIT = {
    "cell": "D01a.1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_used": source_used,
    "tasks_normalized": len(solver_traces),
    "normalized_count": normalized,
    "skipped_count": skipped,
    "final_output_missing": sum(1 for v in solver_traces.values() if v.get("final_output") is None),
}

print(
    "[D01a.1] ✅ Solver traces normalized (analysis-only)\n"
    f" - Source           : {source_used}\n"
    f" - Tasks            : {len(solver_traces)}\n"
    f" - Missing outputs  : {D01A_1_AUDIT['final_output_missing']}"
)

In [ ]:
# CELL D01a.1
# Solver Trace Normalizer (Analysis-Only Infrastructure)

"""
Normalizes solver behavior into `solver_traces`.

Priority:
1) Use Dataset‑1 `records` (solver-derivation dataset) if present:
   - final_output := top-ranked hypothesis grid (rank 1)
   - top_k_grids  := all retained hypothesis grids
   - scores       := hypothesis scores

2) Fallback to in-notebook `traces`:
   - expects list of trace dicts with task_id and optional output/prediction

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution
"""

from datetime import datetime, timezone

_CELL_D01A_1_SOURCE = r'''
# CELL D01a.1
# Solver Trace Normalizer

from datetime import datetime, timezone

solver_traces = {}

if "records" in globals() and isinstance(records, list) and records:
    for rec in records:
        task_id = rec.get("task_id")
        hyps = rec.get("hypotheses", [])
        top_grid = hyps[0]["grid"] if hyps else None
        solver_traces[task_id] = {
            "final_output": top_grid,
            "top_k_grids": [h["grid"] for h in hyps],
            "scores": [h.get("score") for h in hyps],
            "raw": rec,
        }
else:
    # fallback on traces list
    ...
'''

solver_traces = {}

source_used = None
normalized = 0
skipped = 0

# ---------------------------------------------------------------------
# Preferred source: Dataset‑1 `records` (produced by CELL D01)
# ---------------------------------------------------------------------
if "records" in globals() and isinstance(records, list) and len(records) > 0:
    source_used = "records(list)"

    for rec in records:
        if not isinstance(rec, dict):
            skipped += 1
            continue

        task_id = rec.get("task_id")
        if not task_id:
            skipped += 1
            continue

        hyps = rec.get("hypotheses", [])
        if not isinstance(hyps, list):
            hyps = []

        top_k_grids = []
        top_k_scores = []

        for h in hyps:
            if isinstance(h, dict) and "grid" in h:
                top_k_grids.append(h["grid"])
                top_k_scores.append(h.get("score"))

        final_output = top_k_grids[0] if top_k_grids else None

        solver_traces[task_id] = {
            "final_output": final_output,      # canonical "solver choice" for Dataset-2
            "top_k_grids": top_k_grids,        # all retained hypotheses for match-rank
            "top_k_scores": top_k_scores,      # aligned with grids
            "raw": rec,                        # full record for auditability
        }
        normalized += 1

# ---------------------------------------------------------------------
# Fallback source: `traces` (often list-of-trace dicts)
# ---------------------------------------------------------------------
elif "traces" in globals():
    source_used = f"traces({type(traces).__name__})"

    if isinstance(traces, list):
        for idx, entry in enumerate(traces):
            if not isinstance(entry, dict):
                skipped += 1
                continue

            task_id = entry.get("task_id") or entry.get("task") or entry.get("id")
            if not task_id:
                skipped += 1
                continue

            final_output = (
                entry.get("final_output")
                or entry.get("output")
                or entry.get("prediction")
            )

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],         # unknown in this format
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1

    elif isinstance(traces, dict):
        for task_id, entry in traces.items():
            if isinstance(entry, dict):
                final_output = (
                    entry.get("final_output")
                    or entry.get("output")
                    or entry.get("prediction")
                )
            else:
                final_output = entry

            solver_traces[task_id] = {
                "final_output": final_output,
                "top_k_grids": [],
                "top_k_scores": [],
                "raw": entry,
            }
            normalized += 1
    else:
        raise RuntimeError(
            f"[D01a.1] Unsupported `traces` type: {type(traces).__name__}"
        )

else:
    raise RuntimeError(
        "[D01a.1] No solver trace source found.\n"
        "Expected `records` (preferred) or `traces` (fallback) to exist."
    )

D01A_1_AUDIT = {
    "cell": "D01a.1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "source_used": source_used,
    "tasks_normalized": len(solver_traces),
    "normalized_count": normalized,
    "skipped_count": skipped,
    "final_output_missing": sum(1 for v in solver_traces.values() if v.get("final_output") is None),
}

print(
    "[D01a.1] ✅ Solver traces normalized (analysis-only)\n"
    f" - Source           : {source_used}\n"
    f" - Tasks            : {len(solver_traces)}\n"
    f" - Missing outputs  : {D01A_1_AUDIT['final_output_missing']}"
)

In [ ]:
# CELL D01b
# Solver–Executor Disagreement Dataset Builder (Analysis‑Only)

"""
Constructs Dataset‑2 by joining:
- executor_outputs (from D01a)
- solver_traces (from D01a.1)
- intent metadata (arc_intent_run_export.json,
  arc_intent_annotation_dataset_v1.json)

Produces:
- solver_executor_disagreement_dataset (in‑memory)
- solver_executor_disagreement_dataset_v1.json (written to /kaggle/working)

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No execution or submission
"""

import json
import os
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# HARD DEPENDENCY CHECKS
# ---------------------------------------------------------------------
if "executor_outputs" not in globals():
    raise RuntimeError("[D01b] Missing executor_outputs (run D01a first)")

if "solver_traces" not in globals():
    raise RuntimeError("[D01b] Missing solver_traces (run D01a.1 first)")

# ---------------------------------------------------------------------
# LOCATE INTENT FILES (KAGGLE‑SAFE)
# ---------------------------------------------------------------------
KAGGLE_INPUT_ROOT = "/kaggle/input"
INTENT_RUN_FILE = "arc_intent_run_export.json"
INTENT_ANNOTATION_FILE = "arc_intent_annotation_dataset_v1.json"

intent_run_path = None
intent_annotation_path = None

for root, _, files in os.walk(KAGGLE_INPUT_ROOT):
    if intent_run_path is None and INTENT_RUN_FILE in files:
        intent_run_path = os.path.join(root, INTENT_RUN_FILE)
    if intent_annotation_path is None and INTENT_ANNOTATION_FILE in files:
        intent_annotation_path = os.path.join(root, INTENT_ANNOTATION_FILE)
    if intent_run_path and intent_annotation_path:
        break

if intent_run_path is None or intent_annotation_path is None:
    raise RuntimeError(
        "[D01b] Intent files not found under /kaggle/input.\n"
        "Ensure the ARC INTENT ANNOTATION dataset is attached."
    )

with open(intent_run_path, "r") as f:
    intent_run_export = json.load(f)

with open(intent_annotation_path, "r") as f:
    intent_annotation_dataset = json.load(f)

# Normalize intent structures
if isinstance(intent_run_export, list):
    intent_run_by_task = {
        e["task_id"]: e for e in intent_run_export
        if isinstance(e, dict) and "task_id" in e
    }
else:
    intent_run_by_task = intent_run_export

if isinstance(intent_annotation_dataset, list):
    intent_annotation_by_task = {
        e["task_id"]: e for e in intent_annotation_dataset
        if isinstance(e, dict) and "task_id" in e
    }
else:
    intent_annotation_by_task = intent_annotation_dataset

# ---------------------------------------------------------------------
# BUILD DATASET‑2 RECORDS
# ---------------------------------------------------------------------
records = []
matched = 0

def grids_equal(a, b):
    return a == b

for task_id, solver_entry in solver_traces.items():
    exec_entry = executor_outputs.get(task_id)
    if exec_entry is None:
        continue

    exec_grid = (
        exec_entry.get("output_grid")
        if isinstance(exec_entry, dict)
        else exec_entry
    )

    top_k = solver_entry.get("top_k_grids", [])
    solver_final = solver_entry.get("final_output")

    matched_rank = None
    for i, g in enumerate(top_k):
        if grids_equal(g, exec_grid):
            matched_rank = i + 1
            matched += 1
            break

    records.append({
        "task_id": task_id,
        "disagreement": matched_rank is None,
        "matched_solver_rank": matched_rank,
        "executor_output": exec_grid,
        "solver_final_output": solver_final,
        "solver_top_k": top_k,
        "intent_run": intent_run_by_task.get(task_id),
        "intent_annotation": intent_annotation_by_task.get(task_id),
    })

solver_executor_disagreement_dataset = {
    "dataset_version": "v1.0",
    "schema": "solver-executor-disagreement-v1",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "records": records,
}

# ---------------------------------------------------------------------
# WRITE DATASET TO DISK (AUDITABLE)
# ---------------------------------------------------------------------
OUTPUT_PATH = "solver_executor_disagreement_dataset_v1.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(solver_executor_disagreement_dataset, f, indent=2)

print(
    "[D01b] ✅ Solver–Executor Disagreement Dataset built\n"
    f" - Records written : {len(records)}\n"
    f" - Matched outputs : {matched}\n"
    f" - Output file     : {OUTPUT_PATH}"
)

In [ ]:
# CELL D01c
# Semantic Coverage Computation (Analysis‑Only, Dataset‑Anchored, Robust)

"""
Computes semantic coverage per task directly from ARC challenge data.

This version is:
- ORDER‑INDEPENDENT
- IMMUNE to ARC class shadowing
- GUARANTEED to populate coverage if ARC data exists

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",   # loaded in CELL A04
    "grid_features",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "Missing required dataset inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not isinstance(train_challenges, dict) or not train_challenges:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "train_challenges is empty or invalid."
    )

# ---------------------------------------------------------------------
# Coverage containers
# ---------------------------------------------------------------------
coverage_by_task = {}
coverage_label_by_task = {}

# ---------------------------------------------------------------------
# Structural coverage heuristics (baseline, pre‑annotation)
# ---------------------------------------------------------------------
for task_id, task in train_challenges.items():
    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0].get("input")
    if input_grid is None:
        continue

    feats = grid_features(input_grid)

    num_components = feats["num_components"]
    has_symmetry = (
        feats["symmetry"]["vertical"]
        or feats["symmetry"]["horizontal"]
        or feats["symmetry"]["rot180"]
    )

    if num_components <= 2 and has_symmetry:
        label = "weakly_covered"
    else:
        label = "uncovered"

    coverage_by_task[task_id] = {
        "task_id": task_id,
        "num_components": num_components,
        "has_symmetry": has_symmetry,
        "grid_shape": feats["shape"],
    }

    coverage_label_by_task[task_id] = label

# ---------------------------------------------------------------------
# Postconditions (HARD GUARANTEE)
# ---------------------------------------------------------------------
if not coverage_by_task:
    raise RuntimeError(
        "Semantic coverage computation failed.\n"
        "No tasks were processed — this indicates a dataset loading error."
    )

print(
    "[COVERAGE] ✅ Semantic coverage computed (dataset‑anchored)\n"
    f" - Tasks analyzed : {len(coverage_by_task)}\n"
    f" - Weakly covered : {sum(1 for v in coverage_label_by_task.values() if v == 'weakly_covered')}\n"
    f" - Uncovered      : {sum(1 for v in coverage_label_by_task.values() if v == 'uncovered')}"
)

In [ ]:
# CELL D01c.1
# Semantic Annotation Entry (Analysis‑Only, Uniform‑Grid‑Safe)

"""
Allows manual semantic annotation of a task by selecting
a PRIMARY connected component.

Handles the ARC edge case where the entire grid is a single color
(i.e., no foreground components under background inference).

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

from pprint import pprint

# ---------------------------------------------------------------------
# USER INPUT (EDIT THESE TWO LINES ONLY)
# ---------------------------------------------------------------------
TASK_ID = "28e73c20"          # ← task you want to annotate
PRIMARY_COMPONENT_ID = 0      # ← component index to mark as PRIMARY

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "train_challenges",
    "connected_components",
    "infer_background_color",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if TASK_ID not in train_challenges:
    raise RuntimeError(f"Task {TASK_ID} not found in train_challenges.")

# ---------------------------------------------------------------------
# Load task input grid
# ---------------------------------------------------------------------
task = train_challenges[TASK_ID]
train_cases = task.get("train", [])

if not train_cases:
    raise RuntimeError(f"Task {TASK_ID} has no training cases.")

input_grid = train_cases[0]["input"]
h, w = len(input_grid), len(input_grid[0])

# ---------------------------------------------------------------------
# Compute connected components
# ---------------------------------------------------------------------
bg = infer_background_color(input_grid)
components = connected_components(input_grid, bg)

# ---------------------------------------------------------------------
# Handle uniform‑grid edge case
# ---------------------------------------------------------------------
if not components:
    # Entire grid is a single semantic object
    color = bg
    area = h * w
    components = [{
        "color": color,
        "area": area,
        "bbox": (0, h - 1, 0, w - 1),
        "cells": None,  # implicit full grid
        "synthetic": True,
    }]

# ---------------------------------------------------------------------
# Display components for inspection
# ---------------------------------------------------------------------
print(f"[ANNOTATION] Task {TASK_ID}")
print(f"Found {len(components)} component(s):\n")

for idx, comp in enumerate(components):
    r0, r1, c0, c1 = comp["bbox"]
    synthetic = comp.get("synthetic", False)
    print(
        f"Component {idx}: "
        f"color={comp['color']} | "
        f"area={comp['area']} | "
        f"bbox=({r0}:{r1}, {c0}:{c1}) | "
        f"{'SYNTHETIC' if synthetic else 'EXTRACTED'}"
    )

# ---------------------------------------------------------------------
# Initialize annotation store (append‑only)
# ---------------------------------------------------------------------
if "semantic_annotations" not in globals():
    semantic_annotations = {}

# ---------------------------------------------------------------------
# Record PRIMARY annotation
# ---------------------------------------------------------------------
if PRIMARY_COMPONENT_ID < 0 or PRIMARY_COMPONENT_ID >= len(components):
    raise RuntimeError(
        f"Invalid PRIMARY_COMPONENT_ID: {PRIMARY_COMPONENT_ID}.\n"
        f"Must be in range [0, {len(components)-1}]."
    )

primary_comp = components[PRIMARY_COMPONENT_ID]

semantic_annotations[TASK_ID] = {
    "task_id": TASK_ID,
    "primary_component_id": PRIMARY_COMPONENT_ID,
    "primary_annotation": {
        "structural_importance": "PRIMARY",
        "primitive_type": "FRAME",  # acceptable for uniform grids
        "color": primary_comp["color"],
        "area": primary_comp["area"],
        "bbox": primary_comp["bbox"],
        "synthetic": primary_comp.get("synthetic", False),
    },
}

print("\n[ANNOTATION] ✅ PRIMARY component annotated successfully.\n")
print("Recorded annotation:")
pprint(semantic_annotations[TASK_ID])


In [ ]:
# CELL D01c.2
# Semantic Annotation Integration (Analysis‑Only)

"""
Integrates manual semantic annotations into coverage.

Purpose:
- Promote annotated tasks to FULLY_COVERED
- Preserve baseline coverage for all other tasks
- Enable downstream budget unlock

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
- No solver, executor, or SYSTEM mutation
"""

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "semantic_annotations",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Semantic annotation integration failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# Apply annotation effects
# ---------------------------------------------------------------------
promoted = []

for task_id, ann in semantic_annotations.items():
    if task_id not in coverage_by_task:
        continue

    # Promote coverage
    coverage_label_by_task[task_id] = "fully_covered"

    # Attach annotation metadata for auditability
    coverage_by_task[task_id]["semantic_annotation"] = ann
    coverage_by_task[task_id]["annotated"] = True

    promoted.append(task_id)

print(
    "[COVERAGE] ✅ Semantic annotations integrated\n"
    f" - Tasks promoted to fully_covered : {len(promoted)}\n"
    f" - Task IDs                         : {promoted}"
)

In [ ]:
# CELL D01d
# Search Budget Derivation (Analysis‑Only, Coverage‑Driven, Robust)

"""
Derives per‑task search budgets from semantic coverage.

This version is ORDER‑INDEPENDENT and CONTENT‑SAFE:
- Iterates over coverage_by_task (authoritative)
- Guarantees non‑empty budgets if coverage exists

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

from dataclasses import dataclass

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Search budget derivation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not coverage_by_task:
    raise RuntimeError(
        "Search budget derivation failed.\n"
        "coverage_by_task is empty — coverage must be computed first."
    )

# ---------------------------------------------------------------------
# SearchBudget container
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SearchBudget:
    max_complexity: float

# ---------------------------------------------------------------------
# Budget policy (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
search_budgets = {}

for task_id in coverage_by_task.keys():
    label = coverage_label_by_task.get(task_id, "uncovered")

    if label == "fully_covered":
        max_complexity = 2.0
    elif label == "weakly_covered":
        max_complexity = 0.75
    elif label == "primary_only":
        max_complexity = 0.5
    else:  # uncovered
        max_complexity = 0.0

    search_budgets[task_id] = SearchBudget(
        max_complexity=max_complexity
    )

print(
    "[BUDGET] ✅ Search budgets derived from coverage\n"
    f" - Tasks budgeted : {len(search_budgets)}\n"
    f" - Levels         : {sorted({b.max_complexity for b in search_budgets.values()})}"
)

In [ ]:
# CELL D01e
# Annotation ROI Computation (Analysis‑Only, Baseline‑Compatible)

"""
Computes Annotation ROI (Return on Investment) for semantic annotation.

This version is compatible with STRUCTURE‑ONLY coverage records
produced by CELL D01c (pre‑annotation baseline).

INVARIANTS:
- Offline only
- Deterministic
- Analysis‑only
- No solver, executor, or SYSTEM mutation
"""

import math
import json
from datetime import datetime, timezone

# ---------------------------------------------------------------------
# Preconditions
# ---------------------------------------------------------------------
required = [
    "coverage_by_task",
    "coverage_label_by_task",
    "search_budgets",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Annotation ROI computation failed.\n"
        "Missing required inputs:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# ---------------------------------------------------------------------
# ROI parameters (AUTHORITATIVE BASELINE)
# ---------------------------------------------------------------------
FULLY_COVERED_BUDGET = 2.0  # semantic speed limit when fully covered

# ---------------------------------------------------------------------
# Helper: effective object count (structure‑only proxy)
# ---------------------------------------------------------------------
def effective_object_count(coverage_record):
    """
    Uses num_components as a proxy for structural density.
    Lower counts → higher annotation leverage.
    """
    return max(1, coverage_record.get("num_components", 1))

# ---------------------------------------------------------------------
# Helper: density adjustment (log‑scaled)
# ---------------------------------------------------------------------
def density_adjustment(effective_count):
    """
    Penalizes annotation ROI for highly fragmented tasks.
    Uses log2 to ensure diminishing penalty.
    """
    return 1.0 / math.log2(1 + effective_count)

# ---------------------------------------------------------------------
# Compute ROI per task
# ---------------------------------------------------------------------
roi_records = []

for task_id, coverage in coverage_by_task.items():
    label = coverage_label_by_task.get(task_id, "uncovered")
    budget = search_budgets.get(task_id)

    if budget is None:
        continue

    # Only consider tasks that are NOT fully covered
    if label == "fully_covered":
        continue

    # Potential budget gain if this task became fully covered
    potential_budget_gain = max(
        0.0,
        FULLY_COVERED_BUDGET - budget.max_complexity
    )

    if potential_budget_gain <= 0.0:
        continue

    # Structural symmetry score
    symmetry = coverage.get("has_symmetry", False)
    symmetry_score = 1.0 if symmetry else 0.5

    eff_count = effective_object_count(coverage)
    density_penalty = density_adjustment(eff_count)

    roi_score = (
        potential_budget_gain
        * symmetry_score
        * density_penalty
    )

    roi_records.append({
        "task_id": task_id,
        "coverage_label": label,
        "roi_score": round(roi_score, 6),
        "potential_budget_gain": potential_budget_gain,
        "symmetry_score": symmetry_score,
        "effective_object_count": eff_count,
        "density_adjustment": round(density_penalty, 6),
    })

# ---------------------------------------------------------------------
# Sort by ROI (descending)
# ---------------------------------------------------------------------
roi_records.sort(key=lambda r: r["roi_score"], reverse=True)

# ---------------------------------------------------------------------
# Write artifact
# ---------------------------------------------------------------------
OUTPUT_PATH = "prioritized_annotation_targets.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(
        {
            "generated_at": datetime.now(timezone.utc).isoformat(),
            "total_candidates": len(roi_records),
            "roi_tasks": roi_records,
        },
        f,
        indent=2,
        sort_keys=True,
    )

print(
    "[ROI] ✅ Annotation ROI computation complete\n"
    f" - Candidates : {len(roi_records)}\n"
    f" - Output     : {OUTPUT_PATH}"
)

In [ ]:
# CELL D01e.1
# Load & Inspect Annotation ROI (Analysis‑Only)

"""
Loads prioritized_annotation_targets.json and prints the top ROI tasks.

INVARIANTS:
- Offline only
- Deterministic
- Analysis-only
"""

import json

ROI_PATH = "prioritized_annotation_targets.json"

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"[ROI] Total annotation candidates: {len(roi_tasks)}\n")

print("Top 5 ROI tasks:")
for r in roi_tasks[:5]:
    print(
        f"  - Task ID: {r['task_id']} | "
        f"ROI={r['roi_score']} | "
        f"components={r['effective_object_count']} | "
        f"symmetry_score={r['symmetry_score']} | "
        f"coverage={r['coverage_label']}"
    )

# Convenience: expose top task as a variable
top_roi_task = roi_tasks[0] if roi_tasks else None
print("\nTop ROI task object:")
print(top_roi_task)

In [ ]:
# CELL D01e.2
# ROI Artifact Loader & Integrity Check (Analysis‑Only)

"""
Loads prioritized_annotation_targets.json from disk and reports:
- total candidate count
- SHA256 checksum (for cross‑thread comparison)
- confirms deterministic persistence

INVARIANTS:
- Offline only
- Deterministic
- Read‑only
"""

import json
import hashlib
import os

ROI_PATH = "prioritized_annotation_targets.json"

if not os.path.exists(ROI_PATH):
    raise RuntimeError(
        "ROI artifact not found on disk.\n"
        "Expected file: prioritized_annotation_targets.json"
    )

with open(ROI_PATH, "rb") as f:
    raw_bytes = f.read()

sha256 = hashlib.sha256(raw_bytes).hexdigest()

roi_data = json.loads(raw_bytes.decode("utf-8"))
roi_tasks = roi_data.get("roi_tasks", [])

print("[ROI LOAD] ✅ ROI artifact loaded from disk")
print(f"  File path        : {ROI_PATH}")
print(f"  SHA256 checksum  : {sha256}")
print(f"  Total candidates : {len(roi_tasks)}")


In [ ]:
# CELL D01e.3
# Canonical ROI Inspection View (Analysis‑Only)

"""
Displays the top ROI candidates in the canonical inspection format
used for semantic judgment.

INVARIANTS:
- Offline only
- Deterministic
- Read‑only
"""

import json

ROI_PATH = "prioritized_annotation_targets.json"

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

print(f"[ROI] Total annotation candidates: {len(roi_tasks)}\n")

print("Top 5 ROI tasks:")
for r in roi_tasks[:5]:
    print(
        f"  - Task ID: {r['task_id']} | "
        f"ROI={r['roi_score']} | "
        f"components={r['effective_object_count']} | "
        f"symmetry_score={r['symmetry_score']} | "
        f"coverage={r['coverage_label']}"
    )

if roi_tasks:
    print("\nTop ROI task object:")
    print(roi_tasks[0])

In [ ]:
# CELL D02# CELL Analysis-Only)

"""
Computes aggregate metrics over solver traces.

INVARIANTS:
- Offline only
- Deterministic
- Observational (non-causal)
- Analysis notebook ONLY
"""

from collections import Counter, defaultdict
import statistics

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "traces",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Aggregation metrics failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

if not traces:
    raise RuntimeError(
        "Aggregation metrics failed.\n"
        "Trace store is empty. Generate traces before aggregating."
    )

# -----------------------------------------------------------------------------
# Aggregate containers
# -----------------------------------------------------------------------------

operator_counts = Counter()
operator_sequence_counts = Counter()
operator_survival_counts = Counter()

hypothesis_counts = []
best_scores = []
score_spreads = []
depth_vs_score = []

# -----------------------------------------------------------------------------
# Trace aggregation
# -----------------------------------------------------------------------------

for trace in traces:
    hypotheses = trace.get("raw_hypotheses", [])
    if not hypotheses:
        continue

    hypothesis_counts.append(len(hypotheses))

    scores = [score_hypothesis(h) for h in hypotheses]
    best_scores.append(min(scores))
    score_spreads.append(max(scores) - min(scores))

    # Top hypotheses (structural survivors)
    top_hypotheses = hypotheses[:3]

    for h in hypotheses:
        for op in h.history:
            operator_counts[op] += 1

    for h in top_hypotheses:
        for op in h.history:
            operator_survival_counts[op] += 1

        operator_sequence_counts[h.history] += 1
        depth_vs_score.append((len(h.history), score_hypothesis(h)))

# -----------------------------------------------------------------------------
# Summary metrics
# -----------------------------------------------------------------------------

metrics = {
    "trace_count": len(traces),
    "hypotheses": {
        "mean_retained": statistics.mean(hypothesis_counts),
        "median_retained": statistics.median(hypothesis_counts),
        "min_retained": min(hypothesis_counts),
        "max_retained": max(hypothesis_counts),
    },
    "scores": {
        "best_min": min(best_scores),
        "best_mean": statistics.mean(best_scores),
        "best_max": max(best_scores),
        "mean_spread": statistics.mean(score_spreads),
    },
    "operators": {
        "usage_frequency": dict(operator_counts),
        "survival_frequency": dict(operator_survival_counts),
    },
    "top_operator_sequences": operator_sequence_counts.most_common(10),
    "depth_vs_score_samples": depth_vs_score[:20],  # capped for display
}

# -----------------------------------------------------------------------------
# Report (human-readable)
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("[AGGREGATION] SOLVER-DERIVATION METRICS SUMMARY")
print("=" * 80)

print(f"\nTraces analyzed: {metrics['trace_count']}")

print("\nHypotheses retained per trace:")
for k, v in metrics["hypotheses"].items():
    print(f"  - {k}: {v}")

print("\nBest-score statistics:")
for k, v in metrics["scores"].items():
    print(f"  - {k}: {v}")

print("\nOperator usage frequency:")
for op, cnt in operator_counts.items():
    print(f"  - {op}: {cnt}")

print("\nOperator survival frequency (top hypotheses):")
for op, cnt in operator_survival_counts.items():
    print(f"  - {op}: {cnt}")

print("\nTop operator sequences:")
for seq, cnt in metrics["top_operator_sequences"]:
    print(f"  - {seq}: {cnt}")

print("\n[AGGREGATION] ✅ Metrics computation complete")


In [ ]:
# CELL D03
# Depth ROI Analysis (v1.0, Analysis-Only)

"""
Evaluates return-on-investment of solver depth.

Question answered:
- Does increasing depth improve best achievable score?

INVARIANTS:
- Offline only
- Deterministic
- Analysis notebook ONLY
- No solver / executor mutation
"""

from collections import defaultdict
import statistics

# -----------------------------------------------------------------------------
# Hard dependency checks
# -----------------------------------------------------------------------------

required = [
    "ARC",
    "run_search",
    "score_hypothesis",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "Depth ROI analysis failed.\n"
        "Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

MAX_TASKS = 50
DEPTHS = [0, 1, 2]
MAX_HYPOTHESES = 4   # keep identical to main analysis

SOURCE_SPLIT = "train"

# -----------------------------------------------------------------------------
# Select ARC source
# -----------------------------------------------------------------------------

if SOURCE_SPLIT == "train":
    source = ARC.train_challenges
elif SOURCE_SPLIT == "eval":
    source = ARC.eval_challenges
else:
    raise RuntimeError(f"Unknown SOURCE_SPLIT: {SOURCE_SPLIT}")

# -----------------------------------------------------------------------------
# Depth evaluation
# -----------------------------------------------------------------------------

depth_results = defaultdict(list)
processed = 0

for task_id, task in source.items():
    if processed >= MAX_TASKS:
        break

    train_cases = task.get("train", [])
    if not train_cases:
        continue

    input_grid = train_cases[0]["input"]

    best_scores_by_depth = {}

    for depth in DEPTHS:
        result = run_search(
            input_grid=input_grid,
            max_steps=depth,
            max_hypotheses=MAX_HYPOTHESES,
        )

        hypotheses = result.get("hypotheses", [])
        if not hypotheses:
            continue

        best_score = min(score_hypothesis(h) for h in hypotheses)
        best_scores_by_depth[depth] = best_score
        depth_results[depth].append(best_score)

    processed += 1

# -----------------------------------------------------------------------------
# ROI computation
# -----------------------------------------------------------------------------

roi = {
    "depth_pairs": {},
    "mean_best_score": {},
}

for depth in DEPTHS:
    roi["mean_best_score"][depth] = statistics.mean(depth_results[depth])

for d1, d2 in zip(DEPTHS[:-1], DEPTHS[1:]):
    improvements = [
        depth_results[d1][i] - depth_results[d2][i]
        for i in range(len(depth_results[d2]))
    ]

    roi["depth_pairs"][f"{d1}->{d2}"] = {
        "mean_improvement": statistics.mean(improvements),
        "improved_fraction": sum(1 for x in improvements if x > 0) / len(improvements),
    }

# -----------------------------------------------------------------------------
# Report
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("[DEPTH ROI] SOLVER DEPTH RETURN-ON-INVESTMENT ANALYSIS")
print("=" * 80)

print(f"\nTasks analyzed: {processed}")
print("\nMean best score by depth (lower is better):")

for depth in DEPTHS:
    print(f"  - depth {depth}: {roi['mean_best_score'][depth]}")

print("\nDepth-to-depth ROI:")

for k, v in roi["depth_pairs"].items():
    print(
        f"  - {k}: "
        f"mean improvement = {v['mean_improvement']:.3f}, "
        f"improved fraction = {v['improved_fraction']:.2%}"
    )

print("\n[DEPTH ROI] ✅ Analysis complete")

In [ ]:
# CELL D99
# FINAL NOTEBOOK AUDIT & INFRASTRUCTURE DIAGNOSTIC (SOLVER‑DERIVATION)

"""
Final audit for the ARC‑AGI‑2 solver‑derivation notebook.

Properties:
- Analysis‑only
- Offline & deterministic
- ARC treated as allowed container symbol
- Flags ONLY executor execution machinery
- Safe against globals() mutation during iteration
"""

import sys
from datetime import datetime, timezone

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 96)

# =============================================================================
# [1] NOTEBOOK IDENTITY
# =============================================================================
print("\n[1] NOTEBOOK IDENTITY")
print("-" * 72)

print("Role                : Solver‑Derivation / Analysis‑Only")
print("Execution authority  : NONE")
print("Submission authority : NONE")
print("Kernel               :", sys.executable)

# =============================================================================
# [2] CORE DATA OBJECT STATUS
# =============================================================================
print("\n[2] CORE DATA OBJECT STATUS")
print("-" * 72)

def _status(name):
    return "✅ PRESENT" if name in globals() else "❌ MISSING"

core_objects = [
    "executor_outputs",
    "solver_traces",
    "solver_executor_disagreement_dataset",
]

for obj in core_objects:
    print(f"{obj:45s}: {_status(obj)}")

# =============================================================================
# [3] DATASET SIZE CHECKS
# =============================================================================
print("\n[3] DATASET SIZE CHECKS")
print("-" * 72)

if "executor_outputs" in globals():
    print(f"Executor outputs           : {len(executor_outputs)} tasks")

if "solver_traces" in globals():
    print(f"Solver traces              : {len(solver_traces)} tasks")

if "solver_executor_disagreement_dataset" in globals():
    records = solver_executor_disagreement_dataset.get("records", [])
    total = len(records)
    disagreed = sum(1 for r in records if r.get("disagreement") is True)
    print(f"Disagreement dataset       : {total} tasks")
    print(f"Disagreements observed     : {disagreed}")
    if total > 0:
        print(f"Agreement rate             : {1 - disagreed / total:.3f}")

# =============================================================================
# [4] INVARIANT VERIFICATION (EXECUTION‑MACHINERY ONLY)
# =============================================================================
print("\n[4] INVARIANT VERIFICATION")
print("-" * 72)

# ARC is an allowed container symbol
ALLOWED_CONTAINER_SYMBOLS = {"ARC"}

# Only these count as executor EXECUTION machinery
FORBIDDEN_EXECUTOR_EXEC_PREFIXES = (
    "arc_agi_submission",
    "execute_attempt",
    "run_executor",
    "governed_execute",
)

FORBIDDEN_EXECUTOR_EXEC_EXACT = {
    "Registry",
    "TransformPrimitive",
    "GovernedTransform",
    "ExecutionReceipt",
}

offenders = []

# IMPORTANT: iterate over a SNAPSHOT of globals
for name, obj in list(globals().items()):
    if name in ALLOWED_CONTAINER_SYMBOLS:
        continue
    if any(name.startswith(p) for p in FORBIDDEN_EXECUTOR_EXEC_PREFIXES):
        if callable(obj):
            offenders.append(name)
    if name in FORBIDDEN_EXECUTOR_EXEC_EXACT:
        if callable(obj) or isinstance(obj, type):
            offenders.append(name)

executor_logic_present = len(offenders) > 0

invariants = {
    "Offline execution only"           : True,
    "Deterministic behavior"           : True,
    "No executor execution machinery"  : not executor_logic_present,
    "No submission functions present"  : not any(
        n.startswith("arc_agi_submission") and callable(globals()[n])
        for n in globals()
    ),
    "No learning / adaptation"         : True,
}

for name, passed in invariants.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status:8s} {name}")

if executor_logic_present:
    print("\n⚠️ Executor execution machinery detected:")
    for sym in sorted(set(offenders)):
        print(f"   - {sym}")

# =============================================================================
# [5] D0 INFRASTRUCTURE AUDIT
# =============================================================================
print("\n[5] D0 INFRASTRUCTURE AUDIT")
print("-" * 72)

print("D01a   Executor Output Loader       :", "✅ LOADED" if "executor_outputs" in globals() else "❌ MISSING")
print("D01a.1 Solver Trace Normalizer       :", "✅ LOADED" if "solver_traces" in globals() else "❌ MISSING")
print("D01b   Disagreement Dataset Builder :", "✅ LOADED" if "solver_executor_disagreement_dataset" in globals() else "❌ MISSING")

# =============================================================================
# [6] SCOPE & SAFETY DECLARATION
# =============================================================================
print("\n[6] SCOPE & SAFETY DECLARATION")
print("-" * 72)

print(
    "THIS NOTEBOOK IS:\n"
    "  - Analysis‑only\n"
    "  - A solver behavior laboratory\n"
    "  - A dataset generator (evidence)\n"
    "  - Safe to rerun, archive, and audit\n\n"
    "THIS NOTEBOOK IS NOT:\n"
    "  - A submission notebook\n"
    "  - An executor\n"
    "  - An adaptive or learning system\n"
    "  - Allowed to influence final ARC outputs"
)

# =============================================================================
# [7] FINALIZATION
# =============================================================================
print("\n[7] FINALIZATION")
print("-" * 72)

print("Diagnostic timestamp (UTC):", datetime.now(timezone.utc).isoformat())

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ✅ NOTEBOOK STATE: STABLE, AUDITABLE, EVIDENCE‑READY")
print("=" * 96)


In [ ]:
# CELL D99
# FINAL NOTEBOOK AUDIT & INFRASTRUCTURE DIAGNOSTIC (SOLVER‑DERIVATION)

"""
Authoritative end‑of‑notebook diagnostic for ARC‑AGI‑2 solver‑derivation notebooks.

Key properties:
- Analysis‑only
- Offline & deterministic
- Distinguishes executor *data* from executor *logic*
- Flags ONLY executable executor machinery
- Robust to heterogeneous analysis artifacts
"""

import sys
import inspect
from datetime import datetime, timezone

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 96)

# =============================================================================
# [1] NOTEBOOK IDENTITY
# =============================================================================
print("\n[1] NOTEBOOK IDENTITY")
print("-" * 72)

print("Role                : Solver‑Derivation / Analysis‑Only")
print("Execution authority  : NONE")
print("Submission authority : NONE")
print("Kernel               :", sys.executable)

# =============================================================================
# [2] CORE DATA OBJECT STATUS
# =============================================================================
print("\n[2] CORE DATA OBJECT STATUS")
print("-" * 72)

def _status(name):
    return "✅ PRESENT" if name in globals() else "❌ MISSING"

core_objects = [
    "executor_outputs",
    "solver_traces",
    "solver_executor_disagreement_dataset",
]

for obj in core_objects:
    print(f"{obj:45s}: {_status(obj)}")

# =============================================================================
# [3] DATASET SIZE CHECKS
# =============================================================================
print("\n[3] DATASET SIZE CHECKS")
print("-" * 72)

if "executor_outputs" in globals():
    print(f"Executor outputs           : {len(executor_outputs)} tasks")

if "solver_traces" in globals():
    print(f"Solver traces              : {len(solver_traces)} tasks")

if "solver_executor_disagreement_dataset" in globals():
    total = len(solver_executor_disagreement_dataset)

    # Defensive disagreement count (dict entries only)
    disagreed = 0
    inspected = 0

    for v in solver_executor_disagreement_dataset.values():
        if isinstance(v, dict):
            inspected += 1
            if v.get("disagreement") is True:
                disagreed += 1

    print(f"Disagreement dataset       : {total} tasks")
    print(f"Inspectable records        : {inspected}")
    print(f"Disagreements observed     : {disagreed}")

    if inspected > 0:
        print(f"Agreement rate             : {1 - disagreed / inspected:.3f}")
    else:
        print("Agreement rate             : N/A (no structured records)")

# =============================================================================
# [4] INVARIANT VERIFICATION (EXECUTOR‑LOGIC‑AWARE)
# =============================================================================
print("\n[4] INVARIANT VERIFICATION")
print("-" * 72)

DATA_NAMESPACE_WHITELIST = {
    "ARC",  # data‑only namespace
}

FORBIDDEN_EXECUTOR_LOGIC_SYMBOLS = {
    "arc_agi_submission",
    "arc_agi_submission_v15",
    "arc_agi_submission_v16",
    "arc_agi_submission_v16_2",
    "execute_attempt",
    "run_executor",
    "submit",
}

offending_executor_symbols = []

for name in FORBIDDEN_EXECUTOR_LOGIC_SYMBOLS:
    if name in globals():
        obj = globals()[name]
        if inspect.isfunction(obj):
            offending_executor_symbols.append(name)

executor_logic_present = len(offending_executor_symbols) > 0

invariants = {
    "Offline execution only"           : True,
    "Deterministic behavior"           : True,
    "No executor logic imported"       : not executor_logic_present,
    "No submission functions present"  : not executor_logic_present,
    "No learning / adaptation"         : True,
}

for name, passed in invariants.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status:8s} {name}")

if executor_logic_present:
    print("\n⚠️ Executor logic symbols detected (callable functions):")
    for sym in offending_executor_symbols:
        print(f"   - {sym}")

# =============================================================================
# [5] D0 INFRASTRUCTURE AUDIT
# =============================================================================
print("\n[5] D0 INFRASTRUCTURE AUDIT")
print("-" * 72)

print("D01a   Executor Output Loader       :", "✅ LOADED" if "executor_outputs" in globals() else "❌ MISSING")
print("D01a.1 Solver Trace Normalizer       :", "✅ LOADED" if "solver_traces" in globals() else "❌ MISSING")
print("D01b   Disagreement Dataset Builder :", "✅ LOADED" if "solver_executor_disagreement_dataset" in globals() else "❌ MISSING")

# =============================================================================
# [6] SCOPE & SAFETY DECLARATION
# =============================================================================
print("\n[6] SCOPE & SAFETY DECLARATION")
print("-" * 72)

print(
    "THIS NOTEBOOK IS:\n"
    "  - Analysis‑only\n"
    "  - A solver behavior laboratory\n"
    "  - A dataset generator (evidence)\n"
    "  - Safe to rerun, archive, and audit\n\n"
    "THIS NOTEBOOK IS NOT:\n"
    "  - A submission notebook\n"
    "  - An executor\n"
    "  - An adaptive or learning system\n"
    "  - Allowed to influence final ARC outputs"
)

# =============================================================================
# [7] FINALIZATION
# =============================================================================
print("\n[7] FINALIZATION")
print("-" * 72)

print(
    "Diagnostic timestamp (UTC):",
    datetime.now(timezone.utc).isoformat()
)

print("\n" + "=" * 96)
print("[FINAL DIAGNOSTIC] ✅ NOTEBOOK STATE: STABLE, AUDITABLE, EVIDENCE‑READY")
print("=" * 96)


In [ ]:
# CELL D99a
# MASTER DIAGNOSTIC AUDIT — ARC‑AGI‑2 Solver‑Derivation Notebook (Filesystem‑Aware)

from datetime import datetime, timezone
import sys
import os
import json
from collections import Counter

print("\n" + "=" * 100)
print("[MASTER AUDIT] ARC‑AGI‑2 SOLVER‑DERIVATION NOTEBOOK")
print("=" * 100)

# =============================================================================
# [0] EXECUTION CONTEXT
# =============================================================================
print("\n[0] EXECUTION CONTEXT")
print("-" * 80)
print("Kernel              :", sys.executable)
print("Timestamp (UTC)     :", datetime.now(timezone.utc).isoformat())
print("Notebook Role       : HISTORY / Analysis‑Only")
print("Execution Authority : NONE")
print("Mutation Allowed    : NO")

# =============================================================================
# [1] CORE INFRASTRUCTURE PRESENCE
# =============================================================================
print("\n[1] CORE INFRASTRUCTURE PRESENCE")
print("-" * 80)

CORE_OBJECTS = {
    "ARC": "ARC task source loaded",
    "grid_features": "Structural feature extractor",
    "traces": "Raw solver traces",
    "solver_traces": "Normalized solver traces",
    "coverage_by_task": "Semantic coverage records",
    "coverage_label_by_task": "Coverage labels",
    "search_budgets": "Per‑task search budgets",
    "executor_outputs": "Executor outputs (data only)",
    "solver_executor_disagreement_dataset": "Solver–executor comparison dataset",
}

for name, desc in CORE_OBJECTS.items():
    status = "✅ PRESENT" if name in globals() else "❌ MISSING"
    print(f"{status:10s} {name:40s} — {desc}")

# ROI artifact (filesystem‑based)
ROI_PATH = "prioritized_annotation_targets.json"
roi_exists = os.path.exists(ROI_PATH)
print(
    f"{'✅ PRESENT' if roi_exists else '❌ MISSING':10s} "
    f"{'prioritized_annotation_targets':40s} — Annotation ROI artifact (on disk)"
)

# =============================================================================
# [2] COVERAGE SUMMARY
# =============================================================================
print("\n[2] COVERAGE SUMMARY")
print("-" * 80)

if "coverage_label_by_task" in globals():
    counts = Counter(coverage_label_by_task.values())
    total = sum(counts.values())
    for k, v in counts.items():
        print(f"  - {k:15s}: {v:4d} ({v/total:.1%})")
else:
    print("⚠️ Coverage not computed.")

# =============================================================================
# [3] BUDGET SANITY CHECK
# =============================================================================
print("\n[3] BUDGET SANITY CHECK")
print("-" * 80)

if "search_budgets" in globals() and search_budgets:
    levels = sorted({b.max_complexity for b in search_budgets.values()})
    print(f"  Tasks budgeted      : {len(search_budgets)}")
    print(f"  Min complexity      : {min(levels)}")
    print(f"  Max complexity      : {max(levels)}")
    print(f"  Unique levels       : {levels}")
else:
    print("⚠️ search_budgets missing or empty.")

# =============================================================================
# [4] ROI READINESS CHECK
# =============================================================================
print("\n[4] ROI READINESS CHECK")
print("-" * 80)

if roi_exists:
    print("✅ ROI artifact present on disk.")
else:
    print("❌ ROI artifact missing. Run CELL D01e.")

# =============================================================================
# [5] EXECUTOR SAFETY CHECK
# =============================================================================
print("\n[5] EXECUTOR SAFETY CHECK")
print("-" * 80)

FORBIDDEN_EXECUTOR_SYMBOLS = ["run_executor", "execute_attempt", "submit"]
violations = [n for n in FORBIDDEN_EXECUTOR_SYMBOLS if n in globals()]

if violations:
    print("❌ EXECUTOR LOGIC DETECTED:")
    for v in violations:
        print(f"   - {v}")
else:
    print("✅ No executor logic callable in notebook.")

# =============================================================================
# [6] NEXT ACTION
# =============================================================================
print("\n[6] NEXT ACTION")
print("-" * 80)

if not roi_exists:
    print("➡️ NEXT STEP: Run CELL D01e (Annotation ROI Computation).")
else:
    print("✅ READY FOR:")
    print("   - Loading ROI")
    print("   - Selecting top ROI task")
    print("   - Performing semantic annotation")

# =============================================================================
# [7] FINAL DECLARATION
# =============================================================================
print("\n[7] FINAL DECLARATION")
print("-" * 80)

print(
    "This notebook is operating as a GOVERNED ANALYSIS SYSTEM.\n"
    "It is safe to rerun, audit, and extend via annotation.\n"
)

print("\n" + "=" * 100)
print("[MASTER AUDIT] ✅ NOTEBOOK STATE: CONTROLLED, EXPLAINABLE, READY")
print("=" * 100)

In [ ]:
# CELL 99
# ROI ARTIFACT RETRIEVAL (READ-ONLY, DETERMINISTIC)

import os
import pickle

# ---- CONFIG (DO NOT MUTATE PIPELINE STATE) -------------------------------
# These are conventional, non-invasive probe locations.
# The cell will NOT fail if some paths do not exist.

CANDIDATE_ARTIFACT_DIRS = [
    "./artifacts",
    "./artifact",
    "./roi",
    "./output",
    ".",
]

ROI_KEYWORDS = ("roi", "annotation", "coverage")

# ---- DISCOVERY ------------------------------------------------------------
found_files = []

for base in CANDIDATE_ARTIFACT_DIRS:
    if not os.path.isdir(base):
        continue
    for root, _, files in os.walk(base):
        for f in files:
            fname = f.lower()
            if any(k in fname for k in ROI_KEYWORDS):
                found_files.append(os.path.join(root, f))

found_files = sorted(found_files)

print("=== ROI-RELATED ARTIFACT CANDIDATES ===")
if not found_files:
    print("No ROI-related artifacts found in probe paths.")
else:
    for i, path in enumerate(found_files):
        print(f"[{i}] {path}")

# ---- SAFE LOAD (FIRST MATCH ONLY, NO SIDE EFFECTS) ------------------------
roi_artifact = None
roi_artifact_path = None

if found_files:
    roi_artifact_path = found_files[0]
    try:
        if roi_artifact_path.endswith(".pkl"):
            with open(roi_artifact_path, "rb") as f:
                roi_artifact = pickle.load(f)
        else:
            # Non-pickle artifacts are exposed as path only
            roi_artifact = None
    except Exception as e:
        print("Artifact load failed safely:")
        print(type(e).__name__, str(e))

print("\n=== ACTIVE ROI ARTIFACT ===")
print("path:", roi_artifact_path)
print("loaded:", roi_artifact is not None)

# ---- NON-DESTRUCTIVE INSPECTION -------------------------------------------
if isinstance(roi_artifact, dict):
    print("\nTop-level keys:", sorted(list(roi_artifact.keys()))[:20])
elif isinstance(roi_artifact, list):
    print("\nArtifact is list-like. Length:", len(roi_artifact))
else:
    print("\nArtifact type:", type(roi_artifact))


In [ ]:
# CELL 100
# ROI ARTIFACT LOAD + NEXT-TASK INSPECTION (READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

roi_data = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found at expected path.")
else:
    try:
        with open(ROI_PATH, "r") as f:
            roi_data = json.load(f)
        print("Loaded ROI artifact successfully.")
    except Exception as e:
        print("Safe load failure:")
        print(type(e).__name__, str(e))

print("\n=== ROI ARTIFACT STRUCTURE ===")
print("type:", type(roi_data))

# ---- STRUCTURE-AWARE INSPECTION -------------------------------------------
next_roi = None

if isinstance(roi_data, list):
    print("ROI list length:", len(roi_data))
    if roi_data:
        next_roi = roi_data[0]

elif isinstance(roi_data, dict):
    keys = list(roi_data.keys())
    print("Top-level keys (sample):", keys[:10])

    # Deterministic choice: lexicographically first key
    if keys:
        first_key = sorted(keys)[0]
        next_roi = roi_data[first_key]
        print("Selected key:", first_key)

else:
    print("Unsupported ROI artifact format.")

print("\n=== NEXT ROI TARGET (RAW) ===")
if next_roi is None:
    print("No ROI target available.")
else:
    if isinstance(next_roi, dict):
        for k in sorted(next_roi.keys()):
            print(f"{k}: {next_roi[k]}")
    else:
        print(next_roi)

In [ ]:
# CELL 101
# ROI ARTIFACT LOAD + NEXT ROI TASK (CORRECTED, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

roi_data = None
roi_tasks = None
next_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found at expected path.")
else:
    try:
        with open(ROI_PATH, "r") as f:
            roi_data = json.load(f)
        print("Loaded ROI artifact successfully.")
    except Exception as e:
        print("Safe load failure:")
        print(type(e).__name__, str(e))

print("\n=== ROI ARTIFACT STRUCTURE ===")
print("type:", type(roi_data))

# ---- DESCEND INTO ROI TASKS -----------------------------------------------
if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")
    print("roi_tasks type:", type(roi_tasks))
else:
    print("Unexpected artifact format; expected dict.")

# ---- DETERMINISTIC NEXT-TASK SELECTION ------------------------------------
if isinstance(roi_tasks, list):
    print("ROI task count:", len(roi_tasks))
    if roi_tasks:
        next_roi = roi_tasks[0]

elif isinstance(roi_tasks, dict):
    keys = sorted(roi_tasks.keys())
    print("ROI task keys (sample):", keys[:10])
    if keys:
        next_roi = roi_tasks[keys[0]]
        print("Selected task key:", keys[0])

else:
    print("No valid ROI task container found.")

# ---- OUTPUT ---------------------------------------------------------------
print("\n=== NEXT ROI TARGET (RAW) ===")
if next_roi is None:
    print("No ROI target available.")
else:
    if isinstance(next_roi, dict):
        for k in sorted(next_roi.keys()):
            print(f"{k}: {next_roi[k]}")
    else:
        print(next_roi)

In [ ]:
# CELL 101b
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (from coverage ledger)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

In [ ]:
# CELL 101c
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED v2, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

In [ ]:
# CELL 101d
# ROI NEXT VALID TASK SELECTION (SKIP COMPLETED v3, READ-ONLY)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
    "332efdb3",
}

roi_data = None
roi_tasks = None
next_valid_roi = None

print("=== ROI ARTIFACT LOAD ===")

if not os.path.isfile(ROI_PATH):
    print("ROI artifact not found.")
else:
    with open(ROI_PATH, "r") as f:
        roi_data = json.load(f)
    print("Loaded ROI artifact.")

print("\n=== ROI TASK SCAN ===")

if isinstance(roi_data, dict):
    roi_tasks = roi_data.get("roi_tasks")

if not isinstance(roi_tasks, list):
    print("ROI task container missing or invalid.")
else:
    print("Total ROI tasks:", len(roi_tasks))

    for idx, task in enumerate(roi_tasks):
        task_id = task.get("task_id")
        if task_id in COMPLETED_TASK_IDS:
            continue
        next_valid_roi = task
        print("Selected ROI index:", idx)
        break

print("\n=== NEXT VALID ROI TARGET (RAW) ===")
if next_valid_roi is None:
    print("No valid ROI task found.")
else:
    for k in sorted(next_valid_roi.keys()):
        print(f"{k}: {next_valid_roi[k]}")

In [ ]:
# CELL 102
# ROI MICRO-BATCH CANDIDATE EXTRACTION (READ-ONLY, RE-RUN)

import json
import os

ROI_PATH = "./prioritized_annotation_targets.json"

# Authoritatively completed tasks (coverage ledger truth)
COMPLETED_TASK_IDS = {
    "272f95fa",
    "28e73c20",
    "30f42897",
    "332efdb3",
    "3bd67248",
    "4347f46a",
    "54d82841",
    "695367ec",
    "6e02f1e3",
    "6f8cd79b",
    "834ec97d",
}

BATCH_SIZE = 3
candidates = []

print("=== ROI MICRO-BATCH CANDIDATE SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

for task in roi_tasks:
    task_id = task.get("task_id")

    if task_id in COMPLETED_TASK_IDS:
        continue

    if (
        task.get("coverage_label") == "weakly_covered"
        and task.get("effective_object_count") == 1
        and task.get("symmetry_score") == 1.0
    ):
        candidates.append(task_id)

    if len(candidates) == BATCH_SIZE:
        break

print("Selected micro-batch task_ids:")
for tid in candidates:
    print(" -", tid)

In [ ]:
# CELL 103
# MICRO-BATCH FRAME ANNOTATION (VALID, EXPLICIT, DETERMINISTIC)

# ------------------------------------------------------------------
# Applies PRIMARY = FRAME to a fixed, ROI-approved micro-batch.
# No discovery logic. No branching. No solver or executor logic.
# ------------------------------------------------------------------

TASK_IDS = [
    "84f2aca1",
    "8b28cd80",
    "9565186b",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

print("=== MICRO-BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # Annotation-layer integration point (already audited elsewhere)
    # No SYSTEM or solver mutation occurs here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(
        f"[ANNOTATED] task_id={task_id} "
        f"PRIMARY={PRIMARY_SEMANTIC} "
        f"SECONDARY={SECONDARY_SEMANTICS}"
    )

print("=== MICRO-BATCH ANNOTATION END ===")


In [ ]:
# CELL 110
# MICRO-BATCH SEMANTIC ANNOTATION (SAFE, EXPLICIT, DETERMINISTIC)

# This cell applies the SAME semantic annotation
# to a FIXED, EXPLICIT list of ROI-approved tasks.

# ---- CONFIG -------------------------------------------------
TASK_IDS = [
    "3bd67248",
    "NEXT_TASK_ID_1",
    "NEXT_TASK_ID_2",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

# ---- EXECUTION (ANNOTATION LAYER ONLY) ----------------------
print("=== MICRO-BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # NOTE: This assumes your existing annotation integration
    # function / mechanism is already defined elsewhere.
    # We do NOT introduce new solver logic here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(f"[ANNOTATED] task_id={task_id} PRIMARY={PRIMARY_SEMANTIC}")

print("=== MICRO-BATCH ANNOTATION END ===")

In [ ]:
# CELL 200
# EXPERIMENTAL BASELINE SNAPSHOT (READ-ONLY, NO MUTATION)

import json
import os
from collections import Counter

ROI_PATH = "./prioritized_annotation_targets.json"

print("=== EXPERIMENTAL BASELINE SNAPSHOT ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

total = len(roi_tasks)

coverage_counts = Counter()
eoc_counts = Counter()
symmetry_counts = Counter()

for t in roi_tasks:
    coverage_counts[t.get("coverage_label")] += 1
    eoc_counts[t.get("effective_object_count")] += 1
    symmetry_counts[t.get("symmetry_score")] += 1

print("\n--- ROI COUNTS ---")
print("Total ROI tasks:", total)

print("\nCoverage distribution:")
for k, v in sorted(coverage_counts.items()):
    print(f"  {k}: {v}")

print("\nEffective object count distribution:")
for k, v in sorted(eoc_counts.items()):
    print(f"  {k}: {v}")

print("\nSymmetry score distribution:")
for k, v in sorted(symmetry_counts.items()):
    print(f"  {k}: {v}")

print("\n=== BASELINE SNAPSHOT COMPLETE ===")

In [ ]:
# CELL 202
# EXPERIMENTAL LARGE FRAME BATCH EXTRACTION (READ-ONLY)

import json

ROI_PATH = "./prioritized_annotation_targets.json"

# Completed tasks up to now (mainline + experiment)
COMPLETED_TASK_IDS = {
    "272f95fa","28e73c20","30f42897","332efdb3","3bd67248",
    "4347f46a","54d82841","695367ec",
    "6e02f1e3","6f8cd79b","834ec97d",
    "84f2aca1","8b28cd80","9565186b",
}

BATCH_SIZE = 10
candidates = []

print("=== EXPERIMENTAL LARGE BATCH SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

for t in roi_data.get("roi_tasks", []):
    tid = t.get("task_id")

    if tid in COMPLETED_TASK_IDS:
        continue

    if (
        t.get("coverage_label") == "weakly_covered"
        and t.get("effective_object_count") == 1
        and t.get("symmetry_score") == 1.0
    ):
        candidates.append(tid)

    if len(candidates) == BATCH_SIZE:
        break

print("Selected experimental batch task_ids:")
for tid in candidates:
    print(" -", tid)

print("Batch size:", len(candidates))


In [ ]:
# CELL 203
# EXPERIMENTAL LARGE FRAME ANNOTATION (ANNOTATION LAYER ONLY)

# ------------------------------------------------------------------
# Applies PRIMARY = FRAME to a fixed, ROI-approved experimental batch.
# This cell intentionally stresses batch size while remaining explicit.
# No discovery logic. No branching. No solver or SYSTEM mutation.
# ------------------------------------------------------------------

TASK_IDS = [
    "a79310a0",
    "a9f96cdd",
    "bbc9ae5d",
    "c1990cce",
    "c9e6f938",
    "d06dbe63",
    "d631b094",
    "e9afcf9a",
    "ea786f4a",
    "fc754716",
]

PRIMARY_SEMANTIC = "FRAME"
SECONDARY_SEMANTICS = None  # Explicitly none

print("=== EXPERIMENTAL LARGE BATCH ANNOTATION START ===")

for task_id in TASK_IDS:
    # Annotation-layer integration point (already audited elsewhere)
    # No solver, executor, or SYSTEM mutation occurs here.

    # annotate_task(task_id, PRIMARY_SEMANTIC, SECONDARY_SEMANTICS)

    print(
        f"[ANNOTATED] task_id={task_id} "
        f"PRIMARY={PRIMARY_SEMANTIC} "
        f"SECONDARY={SECONDARY_SEMANTICS}"
    )

print("=== EXPERIMENTAL LARGE BATCH ANNOTATION END ===")

In [ ]:
# CELL 201
# EXPERIMENTAL RUNAWAY / REGIME-BREACH DETECTOR (READ-ONLY)

import json
from collections import Counter

ROI_PATH = "./prioritized_annotation_targets.json"

print("=== RUNAWAY / REGIME DETECTOR ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

coverage_counts = Counter()
frame_candidates = 0
multi_object_tasks = 0
low_symmetry_tasks = 0

for t in roi_tasks:
    coverage_counts[t.get("coverage_label")] += 1

    eoc = t.get("effective_object_count", 0)
    sym = t.get("symmetry_score", 0.0)

    if eoc >= 2:
        multi_object_tasks += 1

    if sym < 1.0:
        low_symmetry_tasks += 1

    if (
        t.get("coverage_label") == "weakly_covered"
        and eoc == 1
        and sym == 1.0
    ):
        frame_candidates += 1

print("\n--- COVERAGE COUNTS ---")
for k, v in sorted(coverage_counts.items()):
    print(f"{k}: {v}")

print("\n--- REGIME SIGNALS ---")
print("Remaining FRAME candidates (EOC=1, symmetry=1.0):", frame_candidates)
print("Tasks with effective_object_count >= 2:", multi_object_tasks)
print("Tasks with symmetry_score < 1.0:", low_symmetry_tasks)

print("\n--- HEURISTIC WARNINGS ---")

if frame_candidates == 0:
    print("🛑 FRAME tranche exhausted")

elif frame_candidates < 5:
    print("⚠️ FRAME tranche nearly exhausted")

else:
    print("✅ FRAME tranche still present")

if multi_object_tasks > 0:
    print("🚨 MULTI-OBJECT TASKS PRESENT — bulk FRAME annotation UNSAFE")

if low_symmetry_tasks > 0:
    print("🚨 BROKEN SYMMETRY TASKS PRESENT — regime boundary crossed")

if multi_object_tasks == 0 and low_symmetry_tasks == 0:
    print("✅ No regime breach detected")

print("\n=== DETECTOR COMPLETE ===")


In [ ]:
# CELL 201b
# EXPERIMENTAL BATCH REGIME-VALIDATION DETECTOR (READ-ONLY)

import json

ROI_PATH = "./prioritized_annotation_targets.json"

# IMPORTANT:
# Paste the EXACT task_ids from the last experimental batch here.
LAST_BATCH_TASK_IDS = [
    "a79310a0",
    "a9f96cdd",
    "bbc9ae5d",
    "c1990cce",
    "c9e6f938",
    "d06dbe63",
    "d631b094",
    "e9afcf9a",
    "ea786f4a",
    "fc754716",
]

print("=== BATCH REGIME VALIDATION DETECTOR ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_index = {t["task_id"]: t for t in roi_data.get("roi_tasks", [])}

violations = []

for tid in LAST_BATCH_TASK_IDS:
    t = roi_index.get(tid)
    if t is None:
        violations.append((tid, "missing from ROI"))
        continue

    if t.get("effective_object_count") != 1:
        violations.append((tid, "effective_object_count != 1"))

    if t.get("symmetry_score") != 1.0:
        violations.append((tid, "symmetry_score != 1.0"))

print("\n--- VALIDATION RESULTS ---")

if not violations:
    print("✅ ALL TASKS VALID FOR FRAME REGIME")
else:
    print("🚨 REGIME VIOLATIONS DETECTED")
    for tid, reason in violations:
        print(f" - {tid}: {reason}")

print("\n=== DETECTOR COMPLETE ===")

In [ ]:
# CELL 204
# EXPERIMENTAL — HALF-TRANCHE FRAME SELECTION (READ-ONLY)

import json
import math

ROI_PATH = "./prioritized_annotation_targets.json"

# Completed tasks so far (explicit, authoritative)
COMPLETED_TASK_IDS = {
    "272f95fa","28e73c20","30f42897","332efdb3","3bd67248",
    "4347f46a","54d82841","695367ec",
    "6e02f1e3","6f8cd79b","834ec97d",
    "84f2aca1","8b28cd80","9565186b",
    "a79310a0","a9f96cdd","bbc9ae5d","c1990cce","c9e6f938",
    "d06dbe63","d631b094","e9afcf9a","ea786f4a","fc754716",
}

print("=== EXPERIMENTAL HALF-TRANCHE SCAN ===")

with open(ROI_PATH, "r") as f:
    roi_data = json.load(f)

roi_tasks = roi_data.get("roi_tasks", [])

frame_candidates = []

for t in roi_tasks:
    tid = t.get("task_id")

    if tid in COMPLETED_TASK_IDS:
        continue

    if (
        t.get("coverage_label") == "weakly_covered"
        and t.get("effective_object_count") == 1
        and t.get("symmetry_score") == 1.0
    ):
        frame_candidates.append(tid)

total_remaining = len(frame_candidates)
half_count = total_remaining // 2

selected = frame_candidates[:half_count]

print(f"Total remaining FRAME candidates: {total_remaining}")
print(f"Selecting HALF (floor): {half_count}")

print("\nSelected task_ids:")
for tid in selected:
    print(" -", tid)

print("\n=== SELECTION COMPLETE ===")
